# MS-TCM demo: fits on the FRFR-category dataset

This notebook loads the bundled FRFR-category dataset, runs the MS-TCM fit
(point MLE + 95% bootstrap CIs) and the standard-TCM baseline, and plots
their AIC/BIC comparison.

All definitions come from the `ms_tcm` package; this notebook only
composes them (Constitution Principle II: Single Source of Truth).

In [ ]:
import matplotlib.pyplot as plt

from ms_tcm import load_frfr_category
from ms_tcm.bootstrap import bootstrap_ci

ds = load_frfr_category()
print(f'{ds.num_participants} participants, {ds.num_lists_per_participant} lists/pt, '
      f'{ds.num_words_per_list} words/list, feature_dim={ds.feature_dim}')

## Fit MS-TCM and standard-TCM

Demo uses a small bootstrap count for speed; full-scale fits (1000 bootstraps)
are produced via the `ms-tcm fit` CLI.

In [ ]:
mstcm = bootstrap_ci(ds, n_bootstraps=20, n_restarts=3, seed=42)
print('MS-TCM  log L =', mstcm.log_likelihood, ' AIC =', mstcm.aic)
for name, info in mstcm.parameters.items():
    print(f'  {name}: mle={info["mle"]:.3f}  ci=[{info["ci_lower"]:.3f}, {info["ci_upper"]:.3f}]')

In [ ]:
tcm = bootstrap_ci(ds, n_bootstraps=20, n_restarts=3, seed=42, standard_tcm=True)
print('std TCM log L =', tcm.log_likelihood, ' AIC =', tcm.aic)
for name, info in tcm.parameters.items():
    print(f'  {name}: mle={info["mle"]:.3f}  ci=[{info["ci_lower"]:.3f}, {info["ci_upper"]:.3f}]')

## Figure: AIC / BIC comparison

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3.2))
models = ['MS-TCM', 'standard TCM']
ax[0].bar(models, [mstcm.aic, tcm.aic]); ax[0].set_ylabel('AIC')
ax[1].bar(models, [mstcm.bic, tcm.bic]); ax[1].set_ylabel('BIC')
fig.tight_layout(); plt.show()